<a href="https://colab.research.google.com/github/rkahrya1311/amazon-ml-challenge-2026/blob/kishor%2Fmodel/kishor_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# FINAL ML MATCHER FOR ENTITY RESOLUTION
# ============================================================

import pandas as pd
import numpy as np
import joblib

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    confusion_matrix
)


# ============================================================
# 1. MODEL
# ============================================================

MODEL_FILE = "entity_matching_logistic_regression.pkl"
TRAINING_CODE_FILE = "ml_entity_matching.py"

model = Pipeline([

    ("imputer", SimpleImputer(strategy="median")),

    ("scaler", StandardScaler()),

    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])


# ============================================================
# 2. UPLOAD ARUN FEATURE DATASET
# ============================================================

print("=" * 60)
print("UPLOAD ML FEATURE DATASET")
print("=" * 60)

uploaded = files.upload()

file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)

print("\nDataset filename:", file_name)
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())


# ============================================================
# 3. SET TARGET COLUMN
# ============================================================

TARGET_COLUMN = "label"

if TARGET_COLUMN not in df.columns:
    raise ValueError(
        "ERROR: 'label' column not found in the uploaded dataset."
    )


# ============================================================
# 4. SEPARATE FEATURES AND TARGET
# ============================================================

X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]


# Use only numeric features
X = X.select_dtypes(include=["number"])


print("\nFeature columns used by model:")
print(X.columns.tolist())

print("\nLabel distribution:")
print(y.value_counts().sort_index())

positive_samples = int((y == 1).sum())
negative_samples = int((y == 0).sum())

print("\nPositive samples:", positive_samples)
print("Negative samples:", negative_samples)


# ============================================================
# 5. TRAIN / VALIDATION SPLIT
# ============================================================

X_train, X_valid, y_train, y_valid = train_test_split(

    X,
    y,

    test_size=0.20,

    random_state=42,

    stratify=y
)


print("\n" + "=" * 60)
print("VALIDATION SPLIT")
print("=" * 60)

print("Training rows   :", len(X_train))
print("Validation rows :", len(X_valid))

print("\nTraining label counts:")
print(y_train.value_counts().sort_index())

print("\nValidation label counts:")
print(y_valid.value_counts().sort_index())

print("\nSplit type:")
print("Random row-level split")
print("NOT split by Source 1 entity_id")


# ============================================================
# 6. TRAIN MODEL
# ============================================================

print("\n" + "=" * 60)
print("MODEL TRAINING")
print("=" * 60)

model.fit(X_train, y_train)

print("Model training completed!")


# ============================================================
# 7. VALIDATION PROBABILITIES
# ============================================================

probabilities = model.predict_proba(X_valid)[:, 1]


# ============================================================
# 8. TEST THRESHOLDS USING F0.5
# ============================================================

threshold_results = []

best_threshold = 0.50
best_f05 = -1

for threshold in np.arange(0.10, 1.00, 0.01):

    predictions = (
        probabilities >= threshold
    ).astype(int)

    precision = precision_score(
        y_valid,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_valid,
        predictions,
        zero_division=0
    )

    f05 = fbeta_score(
        y_valid,
        predictions,
        beta=0.5,
        zero_division=0
    )

    threshold_results.append({

        "threshold": round(float(threshold), 2),

        "precision": precision,

        "recall": recall,

        "f05": f05
    })

    if f05 > best_f05:

        best_f05 = f05

        best_threshold = float(threshold)


threshold_results_df = pd.DataFrame(threshold_results)


# ============================================================
# 9. PRINT THRESHOLD RESULTS
# ============================================================

print("\n" + "=" * 60)
print("THRESHOLD RESULTS")
print("=" * 60)

print(
    threshold_results_df.to_string(
        index=False
    )
)


# ============================================================
# 10. FINAL VALIDATION PREDICTION
# ============================================================

final_predictions = (

    probabilities >= best_threshold

).astype(int)


precision = precision_score(
    y_valid,
    final_predictions,
    zero_division=0
)

recall = recall_score(
    y_valid,
    final_predictions,
    zero_division=0
)

f05 = fbeta_score(
    y_valid,
    final_predictions,
    beta=0.5,
    zero_division=0
)

f1 = f1_score(
    y_valid,
    final_predictions,
    zero_division=0
)

cm = confusion_matrix(
    y_valid,
    final_predictions
)


# ============================================================
# 11. FINAL RESULTS
# ============================================================

print("\n" + "=" * 60)
print("FINAL VALIDATION RESULTS")
print("=" * 60)

print("Best Threshold :", round(best_threshold, 2))
print("Precision      :", round(precision, 4))
print("Recall         :", round(recall, 4))
print("F0.5           :", round(f05, 4))
print("F1             :", round(f1, 4))

print("\nConfusion Matrix:")
print(cm)

print("\nIMPORTANT:")
print("F0.5 type: PAIR-LEVEL")
print("Validation split: RANDOM ROW-LEVEL")
print("NOT macro-averaged per S1 entity")


# ============================================================
# 12. SAVE MODEL
# ============================================================

model_package = {

    "model": model,

    "threshold": best_threshold,

    "feature_columns": X.columns.tolist(),

    "target_column": TARGET_COLUMN
}


joblib.dump(
    model_package,
    MODEL_FILE
)


print("\n" + "=" * 60)
print("MODEL SAVED")
print("=" * 60)

print("Model file:", MODEL_FILE)


# ============================================================
# 13. INFERENCE FUNCTION
# ============================================================

def predict_matches(candidate_features):

    """
    Input:
        candidate_features
        DataFrame containing the same feature columns
        used during training.

    Output:
        DataFrame containing match probabilities
        and match predictions.
    """

    package = joblib.load(
        MODEL_FILE
    )

    trained_model = package["model"]

    threshold = package["threshold"]

    feature_columns = package["feature_columns"]


    # Check required columns

    missing_columns = [

        col

        for col in feature_columns

        if col not in candidate_features.columns
    ]


    if missing_columns:

        raise ValueError(
            f"Missing feature columns: {missing_columns}"
        )


    X_test = candidate_features[
        feature_columns
    ]


    probabilities = trained_model.predict_proba(
        X_test
    )[:, 1]


    predictions = (

        probabilities >= threshold

    ).astype(int)


    result = candidate_features.copy()

    result["match_probability"] = probabilities

    result["prediction"] = predictions


    return result


# ============================================================
# 14. EXAMPLE INFERENCE
# ============================================================

print("\n" + "=" * 60)
print("ML INFERENCE READY")
print("=" * 60)

print(
    "Use:"
)

print(
    "result = predict_matches(candidate_features)"
)

print(
    "result contains:"
)

print(
    "match_probability"
)

print(
    "prediction"
)

print(
    "prediction = 1 → MATCH"
)

print(
    "prediction = 0 → NO MATCH"
)

ki
